In [ ]:
import numpy as np
from numpy import sqrt
import pandas as pd
import matplotlib.pyplot as plt

import numbers

In [ ]:
def qadd(a, b):
    return sqrt(a**2 + b**2)


def format(float):
    return f"{float:.3g}"


np.set_printoptions(formatter={"float": format})


class Measurement:
    def __init__(self, value: float | list[float] | np.ndarray, name: str | None = ""):
        value = (
            value.astype(float) if isinstance(value, np.ndarray) else np.array(value, dtype=float)
        )
        if value.ndim == 0:
            self.value = value.reshape((1,))
        elif value.ndim == 1:
            self.value = value
        else:
            raise ValueError(f"Measurements must be of shape (n,). Got {value.shape}.")
        self.name = name
        self._error = None

        # initialize no dependancies as standard, add self if explicit error is defined
        self._dependancies: dict[Measurement, float | np.ndarray] = (
            {}
        )  

    def __repr__(self):
        if any(self.error() != 0):
            return f"Measurement({self.name}, value={self.value}, error={self.error()})"
        else:
            return f"Measurement({self.name}, {self.value.tolist()})"

    def setError(self, error: float | list[float] | np.ndarray):
        error = error if isinstance(error, np.ndarray) else np.array(error)
        if error.shape == tuple():
            self._error = np.full_like(self.value, error)
        elif error.shape == self.value.shape:
            self._error = error
        else:
            raise ValueError(
                f"Error must be scalar or of same shape as value. Expected () or {self.value.shape}, got {error.shape}."
            )
        # (over)write all dependancies when error gets (over)written
        self._dependancies = {self: 1}  
        return self

    def error(self):
        squared = 0
        for mes, diff in self._dependancies.items():
            squared += (diff * mes._error) ** 2
        return sqrt(squared)

    def setName(self, name: str):
        self.name = name
        return self

    def group(self):
        self.name = f"({self.name})"
        return self

    def mean(self):
        syst_error = 1 / sqrt( 
            np.sum(1 / self.error()**2)
        )  # bei konstantem Fehler σ => = σ/\sqrt N
        out_error = sqrt(self.sem() ** 2 + syst_error**2)
        return Measurement([self.value.mean()], name=f"μ_{self.name}").setError(out_error)

    def std(self):
        # Da der Mittelwert aus den N Werten folgt, sind nur N - 1 Differenzen unabhängig. => ddof=1
        return self.value.std(ddof=1)

    def sem(self):
        return self.std() / sqrt(len(self.value))

    def __add__(self, other):
        if not isinstance(other, Measurement):
            other = Measurement(other, name=str(other))

        out_value = self.value + other.value
        out_name = f"{self.name} + {other.name}"
        out = Measurement(out_value, out_name)

        uniques = set(self._dependancies.keys()) | set(other._dependancies.keys())
        for u in uniques:
            out._dependancies[u] = self._dependancies.get(u, 0) + other._dependancies.get(
                u, 0
            )  # operation dependant

        return out

    def __mul__(self, other):
        if not isinstance(other, Measurement):
            other = Measurement(other, name=str(other))

        out_value = self.value * other.value
        out_name = f"{self.name} * {other.name}"
        out = Measurement(out_value, out_name)

        uniques = set(self._dependancies.keys()) | set(other._dependancies.keys())
        for u in uniques:
            out._dependancies[u] = (other.value * self._dependancies.get(u, 0)) + (
                self.value * other._dependancies.get(u, 0)
            )  # operation dependant

        return out

    def __radd__(self, other):
        if not isinstance(other, Measurement):
            other = Measurement(other, name=str(other))

        out = self + other
        out.name = f"{other.name} + {self.name}"
        return out

    def __rmul__(self, other):
        if not isinstance(other, Measurement):
            other = Measurement(other, name=str(other))

        out = self * other
        out.name = f"{other.name} * {self.name}"
        return out

    def __neg__(self):
        out = self * (-1)
        out.name = f"- {self.name}"
        return out

    def __sub__(self, other):
        if not isinstance(other, Measurement):
            other = Measurement(other, name=str(other))

        out = self + (-other)
        out.name = f"{self.name} - {other.name}"
        return out

    def __rsub__(self, other):
        if not isinstance(other, Measurement):
            other = Measurement(other, name=str(other))

        out = other + (-self)
        out.name = f"{other.name}"

In [ ]:


a = {1, 2}
b = {2, 3}

a | b

out = dict()
x = Measurement([5, 1], "x").setError(1)
y = Measurement(1, "y").setError(1)
z = x - x

z._dependancies